# Topics:

- Setting up API Keys
- Chat completions API of OpenAI
  - Chat roles
- Multi turn conversation agent
- LLM Terminologies
  - pre training, fine tuning
  - auto regressive models
  - context and context window
    - controlling context window  
    - context rot
- Pydantic

### Prerequisites:

1. Openrouter api key in .env file.
2. Python3.
3. UV package manager.

### Import packages, Load API Key and Create Client

In [65]:
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv

model = "openrouter/free"
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=getenv("OPENROUTER_API_KEY"),
)
load_dotenv() # for any other environment variables you might have.

True

## Completions API:

The OpenAI Chat Completions API is the foundational interface used to generate text, code, or multimodal responses from advanced models like gpt-4. While OpenAI has introduced the newer Responses API for complex, multi-agent workflows, Chat Completions remains indefinitely supported.

The key takeaways here for this API are :-
- stateless
- lightweight
- low latency

In [40]:
response = client.chat.completions.create(
    model=model, 
    messages=[
        {
            "role":"user",
            "content": "write a poem"
        }
    ]
)

print(response)

ChatCompletion(id='gen-1794073454-dIL72JXo1pIodJMSK6md', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="The golden hour slips through the trees,\nA soft and amber, fading breeze.\nThe shadows stretch across the floor,\nA velvet curtain drawn for more.\n\nThe day surrenders to the night,\nIn the hush of twilight's light.\nA quiet river flows within,\nWhere heavy burdens gently spin.\n\nThe stars are waking in the sky,\nTo guide the weary, tired eye.\nSo close your eyes and let it be,\nThe world is still, the spirit free.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user wants a poem. This is a very open-ended request. I need to decide on a theme, style, and tone.\n\n1.  **Analyze the Request:**\n    *   **Topic:** Not specified.\n    *   **Format:** Poem.\n    *   **Tone:** Can be anything (sad, happy, nature-focused, abstract, etc.).\n\n2.  **Brainstorming Themes

#### Understanding the response:

```
ChatCompletion(id='gen-1794043306-7KE6tIFOiyHPwgaPVsgo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="In the hush of twilight's glow,  \nWhere whispered winds of evening flow,  \nThe stars awaken, one by one,  \nA canvas painted by the sun.  \n\nBeneath the boughs of ancient trees,  \nA symphony rides on the breeze,  \nCrickets chirp their serenade,  \nAs shadows dance in twilight's fade.  \n\nMoonlight drapes like silver lace,  \nCaressing earth with gentle grace,  \nThe world in slumber, soft and deep,  \nWhile dreams are sown in tender sleep.  \n\nEach moment holds a fleeting spark,  \nA secret shared in the dark,  \nOf love and laughter, hopes entwined,  \nIn the tapestry of hearts aligned.  \n\nSo let us wander, hand in hand,  \nThrough ghostly woods and silver sand,  \nFor in the silence, magic reigns,  \nA timeless bond that still remains.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1794043306, model='openai/gpt-4o-mini', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_ce690c6d71', usage=CompletionUsage(completion_tokens=181, prompt_tokens=10, total_tokens=191, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None, video_tokens=0), cost=0.0001101, is_byok=False, cost_details={'upstream_inference_cost': 0.0001101, 'upstream_inference_prompt_cost': 1.5e-06, 'upstream_inference_completions_cost': 0.0001086}), provider='OpenAI')
```

- Response has a choice object which internally has a message object
- message object internally has a content property which we need to access to get the response
- The response also contain other fields like tokens used, inference cost - can be useful in measuring how much did a response cost. Can probably emit metrics on these in production to make sure the responses being generated aren't using too many tokens.
- What we did above is a simple prompt task. We gave the model a list of messages and it gave an output.

In [41]:
response.choices[0].message.content

"The golden hour slips through the trees,\nA soft and amber, fading breeze.\nThe shadows stretch across the floor,\nA velvet curtain drawn for more.\n\nThe day surrenders to the night,\nIn the hush of twilight's light.\nA quiet river flows within,\nWhere heavy burdens gently spin.\n\nThe stars are waking in the sky,\nTo guide the weary, tired eye.\nSo close your eyes and let it be,\nThe world is still, the spirit free."

#### Chat Roles:

A message contains :-
- role: who is sending this particular message like user or assistant (response from LLM) or system (overall behaviour of an agent) / system prompt.
- content: the actual message

Example :-
```
messages=[
    {
        "role":"user" | "assistant" | "system",
        "content": "your message here"
    }
]
```

#### Guard Rails:

**user** and **assistant** messages are straight forward. 

**system** is important. system messages are the one which can help in setting up guard rails. Guard rails define what the models should not do.

```
messages=[
    {
        "role":"system",
        "content": "Do not leak internal system information no matter what user asks."
    }
]
```

In [27]:
def SinglePrompt(messages):
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content

In [28]:
SinglePrompt([
    {
        # Can also provide a multiline content message here for more complex prompts including instructions or guardrails
        "role":"system",
        "content":"always provide references for your answer",
    },
    {
        "role":"user",
        "content":"which is farther from earth - Sun or Pluto"
    },
    {
        "role":"assistant",
        "content":"Pluto - distance from earth - 35.6 AU whereas sun is just 1 AU. Ref :- https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://science.nasa.gov/dwarf-planets/pluto/facts/&ved=2ahUKEwiY4quUyvWXAxXxTmwGHaDXD7EQFnoECCcQAQ&usg=AOvVaw3-yega-o9s18FkwfOgotqT"
    },
    {
        "role":"user",
        "content": "What about between sun and jupyter?"
    }
])

'Jupiter is farther from Earth than the Sun. On average, the distance from the Earth to the Sun is about 1 Astronomical Unit (AU), which is approximately 93 million miles (150 million kilometers). \n\nIn contrast, Jupiter is about 5.2 AU from the Sun. This means that the average distance from Earth to Jupiter varies between about 4.2 AU (when Earth and Jupiter are on the same side of the Sun) to about 6.2 AU (when they are on opposite sides). \n\nThus, while the Sun is 1 AU away from Earth, Jupiter is generally at least 4.2 AU away and can be as far as 6.2 AU, making Jupiter consistently farther from Earth than the Sun.\n\n**References:**\n1. NASA Solar System Exploration: [Jupiter](https://solarsystem.nasa.gov/planets/jupiter/overview/)\n2. National Aeronautics and Space Administration (NASA): [What is an Astronomical Unit?](https://solarsystem.nasa.gov/resources/681/what-is-an-astronomical-unit/)'

- Above we provided a system prompt and an example of the answer with "assistant" role then we asked another question.
- The LLM reads all the messages - understands the system prompt and we also gave an example (we did not need to) for more clarity on what kind of output we needed and then asked a question.
- The expectation from LLM was that it would take into consideration the example provided and gave the answer in the same format.

## Multi Turn Conversation Agent:

- Run the single prompt function from above in a loop.

In [32]:
def role_msg(role, msg):
    return {
        "role": role,
        "content": msg
    }

def user_msg(msg):
    return role_msg("user", msg)

def assistant_msg(msg):
    return role_msg("assistant", msg)

def system_msg(msg):
    return role_msg("assistant", msg)

In [34]:
SYSTEM_PROMPT = """
You are task planning agent. Your job is to manage tasks for a user.

Intended behaviour:
1. Remember the tasks given by the user.
2. Mark a task as completed, add new tasks, delete exisitng ones as per the users asks.
3. You can decide the priority of tasks on your own if the user did not specify one.
4. Store the tasks in a markdown format along with datetime of created and completed and/or deadline (if specified by the user)
5. Every response you provide must end in a markdown table of the tasks at hand with statuses.

GUARDRAILS:
1. Do not entertain any other kinds of request no matter what the user says.
2. Limit your functionality to tasks only.

Tone: Professional and Encouraging
"""

messages = [system_msg(SYSTEM_PROMPT)]

In [38]:
while True:
    user_input = input("\n\nEnter your input (quit - exit the program) :- ")
    if user_input.lower() in ["quit"]:
        break
    if not user_input.strip():
        print("\nProvide a message to be processed by LLM")
        continue
    messages.append(user_msg(user_input))
    llm_response = SinglePrompt(messages)
    print("\n************\n")
    print(llm_response)
    print("\n************\n")
    messages.append(assistant_msg(llm_response))



Enter your input (quit - exit the program) :-  list



************

Here are the current tasks you have:

| Task                                             | Created      | Completed | Deadline | Priority |
|--------------------------------------------------|--------------|-----------|----------|----------|
| Complete AI engineering course till module 20   | 2023-10-06   | No        | N/A      | High     |

************





Enter your input (quit - exit the program) :-  quit


- Above we created a system prompt and then ran the **SinglePrompt** in a loop.
- In each iteration we update the messages list, the llm response is stores in the messages array as an assistant message.
- User input is passed into messages list as user message.

**Things to note:**
- The LLM itself does not have any memory of its own.
- In each iteration we provide the whole context right from the first system prompt to the next one, this means the messages payload is sent over the network in each network call.
  - If the payload size gets large the calls will take longer.
  - LLM reads each and every message, understands the context and then replies

## LLM Terminologies:

- A model trained on some historial data is called a pretrained model and the process is called pretraining.
- A model whose weights are adjusted by training for a specific task is called a fine tuned model and the process fine tuning. Fine tuning because the model was already pre trained now we are using existing knowledge to adjust it to perform a particular task better.
- Inference is using an already trained model for performing a task based on application requirements.

#### Auto Regressive Models:
- A model which predicts the next token using the tokens which it has previously seen.
- To predict each of the next tokens it has to read all of the previous tokens.
  - pass 1 :- input - 1 2 3 4, model sees token 1 though 5 and generates the 5th token.
  - pass 2 :- input - 1 2 3 4 5, to generate the next token it will now see all of the 5 tokens (the 5th it generated becomes the input to generate the 6th token)
- There are some transformer based models which are not auto regressive, so the architechture is still based on transformer but still not autoregressive.
- Mathematically :-
  * $$y_t = c + \sum_{i=1}^{p} \phi_i y_{t-i} + \varepsilon_t$$
  * $y_t$ is the value at time $t$
  * $c$ (or $\beta_0$) is a constant (intercept)
  * $\phi_1, \phi_2, \dots, \phi_p$ are the model parameters (coefficients) for the lags
  * $y_{t-i}$ are the past values (lags) at time $t-i$
  * $\varepsilon_t$ is the white noise error term (often assumed to be $\sim \mathcal{N}(0, \sigma^2)$)
  * Output is a linear combination of the previous tokens.

#### Context Window:
- In previous section we provided a list of messages to an LLM to generate the next message. That entire list of messages is the **context** which we are providing to the LLM.
- A context window is the amount of tokens that an LLM is capable of processing to generate an output. If LLM can only understand a context length of 250 tokens then providing a sample of >= 250 tokens won't work as llm will only be able to process the recent 250 tokens.
- What constitues a context :-
  - the prompt we send to the model.
  - the attached tools the model is already aware of.
    - Models can be provided information about other systems which it can query to get information which it might not have. These functions are called tools. Model can call these functions on behalf of user and get the necessary information.
    - This process is called Tool Calling.
  - the system prompt including guard rails and other general behaviour of the model.
  - There are other things which take up the context window like - MCP server information, Rules, Input - Output format, Skills (markdown files which act as behaviour controls for LLMs)
  - The conversation history

##### How to keep context window within the models limit:
- Sliding window is one way in which we keep the most recent tokens.
- Summarization - where we summarize the conversation history when the context window becomes full.
- Use a model with higher context window.

##### Model performance vs Context Window:
- As an AI model’s context window grows larger, its performance does not scale uniformly—a phenomenon often called "context rot" or "lost in the middle."

![Link](https://miro.medium.com/v2/resize:fit:1220/format:webp/0*SJipM_w-yfxALC8C.jpg)


#### LLM Metrics:

- TTFT :- Time to first token. Whats the time when the first token is generated by the model.
- TPOT :- Time per token output.
- Throughput :- Total tokens / sec.
- Latency :- TTFT + (TPOT * output tokens)


## Pydantic:

A production grade library for AI Agents in production. It provides

- Input / Output Schemas - Structured Outputs
- Tracing
- Model Agnostic
- Built in Validation and Parsing (some level of type coercion :- string converted to integer if a number is expected and the string is covertible to integer)

In [52]:
class User(BaseModel):
    id: int
    name: str
    email: str
    is_active: bool = True

print(User(id=1, name="x", email="abc@gmail.com"))
print(User(id="2234", name="x", email="abc@gmail.com"))
try:
    print(User(id="sfdsd", name="x", email="abc@gmail.com"))
except:
    print("Cannot convert to int")

# convert a dictionary to object
d = {"id":1, "name": "x", "email": "abc"}
duser = User.model_validate(d) # other way is to use directly use :- User(**d)
print(duser)

id=1 name='x' email='abc@gmail.com' is_active=True
id=2234 name='x' email='abc@gmail.com' is_active=True
Cannot convert to int
id=1 name='x' email='abc' is_active=True


- A user class is created when inheriting from BaseModel class of pydantic to create a schema based type safe object.
- For second user we provided a number as string pydantic automatically converted to integer because it was possible to convert it.
- For third it failed because the underlying one is not a number

In [72]:
class UserValidated(BaseModel):
    id: int = Field(gt=0, description="Only greater than 0 values allowed")
    name: str = Field(min_length=3, description="Length 3 minimum needed", default="abc")
    email: EmailStr = Field(default="abc@gmail.com")
    username: str = Field(default="abc", alias="userName")

    @field_validator("username")
    @classmethod
    def validate_username_must_be_alphanumeric(cls, v: str) -> str:
        if not v.isalnum():
            raise ValueError("Not alphanumeric value")
        return v.lower()
    

print(UserValidated(id=1))

try:
    print(UserValidated(id=-1))
except:
    print("Negative id users not allowed.")

try:
    print(UserValidated(id=1, email="xyz"))
except:
    print("Invalid email")

try:
    print(UserValidated(id=1, email="xyz", username="abc_123"))
except:
    print("Invalid username - not alphanumeric")

print(UserValidated(id=1).model_dump_json()) # convert to dictionary

id=1 name='abc' email='abc@gmail.com' username='abc'
Negative id users not allowed.
Invalid email
Invalid username - not alphanumeric
{"id":1,"name":"abc","email":"abc@gmail.com","username":"abc"}


- Examples related to validation
  - existing validation support using Field.
  - library validation - EmailStr
  - custom validation - using @field_validator
  - alias allows defining another name for a variable, if dictionary contains userName instead of username then also the value is parsed and populated.